# Languages Without Borders

Which languages span the most countries, and on which continents? This notebook
finds the world's most geographically dispersed languages using `Language.countries`
and `db.languages.filter()`.

**Requirements**
```
pip install "languages-of-the-world[examples]"
```

In [ ]:
%pip install -q pandas plotly matplotlib


In [ ]:
import pandas as pd
import plotly.express as px

import low

## 1 · Load the database

In [ ]:
db = low.LanguagesOfTheWorld()
print(db)

## 2 · Count countries per language

In [ ]:
rows = [
    {
        "language": lang.label,
        "part3": lang.part3,
        "country_count": len(lang.countries),
        "speaker_count": lang.speaker_count,
        "continents": ", ".join(sorted({c.continent.label for c in lang.countries})),
    }
    for lang in db.languages
    if lang.countries
]

df = pd.DataFrame(rows).sort_values("country_count", ascending=False)
df.head(15)

## 3 · Filter to major languages

In [ ]:
major = db.languages.filter(min_speakers=1_000_000)
major_rows = [
    {
        "language": lang.label,
        "part3": lang.part3,
        "country_count": len(lang.countries),
        "speaker_count": lang.speaker_count,
    }
    for lang in major
]

major_df = pd.DataFrame(major_rows).sort_values("country_count", ascending=False)
print(f"Languages with ≥ 1 M speakers: {len(major_df)}")
major_df.head(15)

## 4 · Horizontal bar chart — top cross-border languages

In [ ]:
top20 = major_df.head(20)

fig = px.bar(
    top20,
    x="country_count",
    y="language",
    orientation="h",
    text="country_count",
    labels={"country_count": "Countries", "language": ""},
    title="Major Languages Spanning the Most Countries",
    color_discrete_sequence=["#72b7b2"],
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=600)
fig.update_traces(textposition="outside")
fig.show()

## 5 · Continent distribution for English

In [ ]:
eng = db.languages.get("eng")
cont_counts = {}
for country in eng.countries:
    cont = country.continent.label
    cont_counts[cont] = cont_counts.get(cont, 0) + 1

cont_df = pd.DataFrame([
    {"continent": k, "countries": v} for k, v in sorted(cont_counts.items(), key=lambda x: -x[1])
])

fig2 = px.bar(
    cont_df,
    x="continent",
    y="countries",
    text="countries",
    title=f"Where is {eng.label} spoken? ({len(eng.countries)} countries)",
    labels={"countries": "Countries"},
    color_discrete_sequence=["#4c78a8"],
)
fig2.update_traces(textposition="outside")
fig2.show()

## 6 · Map countries for one language

In [ ]:
import urllib.request, csv, io

swa = db.languages.get("swa")
country_rows = [
    {"iso2": c.code, "country": c.label, "continent": c.continent.label}
    for c in swa.countries
]
swa_df = pd.DataFrame(country_rows)

_UN_M49_CSV = (
    "https://raw.githubusercontent.com/lukes/ISO-3166-Countries-with-Regional-Codes"
    "/master/all/all.csv"
)
with urllib.request.urlopen(_UN_M49_CSV, timeout=20) as r:
    un_text = r.read().decode("utf-8")
alpha2_to_alpha3 = {
    row["alpha-2"]: row["alpha-3"]
    for row in csv.DictReader(io.StringIO(un_text))
    if row["alpha-2"] and row["alpha-3"]
}
swa_df["iso3"] = swa_df["iso2"].map(alpha2_to_alpha3)

fig3 = px.choropleth(
    swa_df.dropna(subset=["iso3"]),
    locations="iso3",
    color="continent",
    hover_name="country",
    title=f"Countries where {swa.label} is spoken",
    projection="natural earth",
)
fig3.update_layout(margin=dict(l=0, r=0, t=40, b=0))
fig3.show()

## 7 · Summary

In [ ]:
print(f"Languages spoken in ≥ 10 countries: {(df['country_count'] >= 10).sum()}")
print(f"Languages spoken on ≥ 3 continents: {sum(1 for r in rows if len(r['continents'].split(', ')) >= 3)}")
print()
print("Top 5 by country count:")
print(df.head(5)[["language", "country_count", "continents"]].to_string(index=False))